# Weekly Study Tracker Program

*Turn raw text rows into validated objects, summarize them, and report the result through one `main()` workflow.*

This integrated program uses the chapter's data structures and core statements together. Prepared strings stand in for values that could later come from `input()` or a file, so the notebook remains fully executable without pausing.

## Problem Definition

### Question

How can a small program convert a weekly study log into reliable session objects, preserve validation errors, and produce one summary report?

### Raw Data

| Line | Text |
| ---: | --- |
| 1 | `Mon, Python, 1.5, yes` |
| 2 | `Tue, Statistics, 2.0, yes` |
| 3 | `Wed, Python, 1.0, no` |
| 4 | `Thu, Data, 2.5, yes` |
| 5 | `Fri, Python, two, yes` |
| 6 | `Sat, Python, 2.0, yes` |

### Constraints

- Every valid row must contain day, subject, numeric hours, and `yes` or `no`.
- Invalid rows must be reported without stopping the remaining work.
- The program must keep parsing, summarization, reporting, and top-level control in separate functions.
- The final result must preserve valid sessions, validation errors, subject totals, completion status, and summary statistics.

## Session Method Map

| Guided Topic | Integrated Use |
| --- | --- |
| Lists | Store raw rows, valid sessions, errors, flags, and report lines. |
| Dictionaries | Accumulate subject hours and return labeled summaries. |
| Tuples | Keep fixed thresholds, error pairs, and multiple returned values. |
| Sets | Validate completion words and collect unique subjects. |
| Booleans and Comparisons | Create completion, validity, `all()`, and `any()` results. |
| Conditionals | Validate fields, guard empty data, and assign a completion label. |
| Loops | Traverse rows with `enumerate()` and accumulate values with `+=`. |
| Functions | Separate parsing, loading, summarization, formatting, and program control. |
| Input and Error Handling | Convert text, `raise ValueError`, and recover with `try`/`except`. |
| Comprehensions | Extract completion flags and unique subject names. |
| Classes | Represent each valid session with state and display behavior. |
| Modules and Program Structure | Import `mean`, define `main()`, and use the main guard. |

## Program Flow

`raw text → parse and validate → StudySession objects → summary dictionary → formatted report`

Each stage receives a value and returns a predictable result to the next stage. Validation failures travel separately as `(line_number, message)` tuples.

## Imports, Constants, and Raw Input

`mean()` comes from the standard library. A set defines the accepted completion words, and a tuple keeps the ordered completion-rate thresholds fixed.

In [1]:
from statistics import mean

COMPLETION_WORDS = {"yes", "no"}
COMPLETION_THRESHOLDS = (80.0, 60.0)

raw_session_lines = [
    "Mon, Python, 1.5, yes",
    "Tue, Statistics, 2.0, yes",
    "Wed, Python, 1.0, no",
    "Thu, Data, 2.5, yes",
    "Fri, Python, two, yes",
    "Sat, Python, 2.0, yes",
]

print("Raw rows:", len(raw_session_lines))
print("Accepted completion words:", sorted(COMPLETION_WORDS))
print("Completion thresholds:", COMPLETION_THRESHOLDS)

Raw rows: 6
Accepted completion words: ['no', 'yes']
Completion thresholds: (80.0, 60.0)


The input is still text at this stage. The fifth row deliberately contains an invalid hours field so the recovery path can be observed.

## Session Class

The class keeps one session's fields together. `__str__()` defines a readable form, and `is_long_session()` provides behavior based on the stored state.

In [2]:
class StudySession:
    def __init__(self, day: str, subject: str, hours: float, completed: bool):
        self.day = day
        self.subject = subject
        self.hours = hours
        self.completed = completed

    def __str__(self):
        status = "complete" if self.completed else "incomplete"
        return f"{self.day} | {self.subject} | {self.hours:.1f} h | {status}"

    def is_long_session(self):
        return self.hours >= 2.0

example_session = StudySession("Mon", "Python", 1.5, True)
print(example_session)
print("Long session:", example_session.is_long_session())

Mon | Python | 1.5 h | complete
Long session: False


The example object owns four instance attributes. Printing it calls `__str__()`, and the method returns a Boolean derived from its current hours.

## Parsing and Validation Function

The parser splits one row, checks its structure, converts hours, validates domain rules, and returns one `StudySession`. Early `raise` statements stop invalid rows with specific messages.

In [3]:
def parse_session(line: str) -> StudySession:
    parts = [part.strip() for part in line.split(",")]
    if len(parts) != 4:
        raise ValueError("expected four comma-separated fields")

    day, subject, hours_text, completed_text = parts
    if not day or not subject:
        raise ValueError("day and subject cannot be empty")
    if completed_text not in COMPLETION_WORDS:
        raise ValueError("completed field must be yes or no")

    try:
        hours = float(hours_text)
    except ValueError:
        raise ValueError("hours must be numeric")

    if hours < 0:
        raise ValueError("hours cannot be negative")

    completed = completed_text == "yes"
    return StudySession(day, subject, hours, completed)

parsed_example = parse_session(raw_session_lines[0])
print("Parsed example:", parsed_example)

Parsed example: Mon | Python | 1.5 h | complete


The valid row becomes a typed object: the hours text becomes a float and the completion text becomes a Boolean. The parser owns all rules for one row.

## Loading All Rows without Stopping

`enumerate(..., start=1)` attaches a readable line number. Each invalid row is caught and stored as a tuple while the loop continues.

In [4]:
def load_sessions(raw_lines):
    sessions = []
    errors = []

    for line_number, line in enumerate(raw_lines, start=1):
        try:
            sessions.append(parse_session(line))
        except ValueError as error:
            errors.append((line_number, str(error)))

    return sessions, errors

loaded_sessions, loading_errors = load_sessions(raw_session_lines)

print("Valid sessions:", len(loaded_sessions))
for session in loaded_sessions:
    print(" ", session)
print("Errors:", loading_errors)

Valid sessions: 5
  Mon | Python | 1.5 h | complete
  Tue | Statistics | 2.0 h | complete
  Wed | Python | 1.0 h | incomplete
  Thu | Data | 2.5 h | complete
  Sat | Python | 2.0 h | complete
Errors: [(5, 'hours must be numeric')]


Five valid objects are preserved. Line 5 is reported as an error, but it does not prevent line 6 from being parsed.

## Integrated Summary

A loop builds subject totals, comprehensions derive reusable collections, and conditionals assign a label from the fixed thresholds. The empty-list guard prevents division by zero and avoids calling `mean()` without data.

In [5]:
def summarize_week(sessions):
    if not sessions:
        return {
            "session_count": 0,
            "total_hours": 0.0,
            "mean_hours": 0.0,
            "completion_rate": 0.0,
            "completion_label": "no data",
            "subjects": [],
            "hours_by_subject": {},
            "all_completed": False,
            "any_incomplete": False,
        }

    hours_by_subject = {}
    for session in sessions:
        hours_by_subject.setdefault(session.subject, 0.0)
        hours_by_subject[session.subject] += session.hours

    completion_flags = [session.completed for session in sessions]
    incomplete_flags = [not flag for flag in completion_flags]
    unique_subjects = {session.subject for session in sessions}
    session_hours = [session.hours for session in sessions]
    completed_count = sum(completion_flags)
    completion_rate = completed_count / len(sessions) * 100

    steady_threshold, developing_threshold = COMPLETION_THRESHOLDS
    if completion_rate >= steady_threshold:
        completion_label = "steady"
    elif completion_rate >= developing_threshold:
        completion_label = "developing"
    else:
        completion_label = "needs review"

    return {
        "session_count": len(sessions),
        "total_hours": round(sum(session_hours), 1),
        "mean_hours": round(mean(session_hours), 1),
        "completion_rate": round(completion_rate, 1),
        "completion_label": completion_label,
        "subjects": sorted(unique_subjects),
        "hours_by_subject": hours_by_subject,
        "all_completed": all(completion_flags),
        "any_incomplete": any(incomplete_flags),
    }

weekly_summary = summarize_week(loaded_sessions)
print("Weekly summary:", weekly_summary)

Weekly summary: {'session_count': 5, 'total_hours': 9.0, 'mean_hours': 1.8, 'completion_rate': 80.0, 'completion_label': 'steady', 'subjects': ['Data', 'Python', 'Statistics'], 'hours_by_subject': {'Python': 4.5, 'Statistics': 2.0, 'Data': 2.5}, 'all_completed': False, 'any_incomplete': True}


The summary stores labeled results in one dictionary. Comprehensions build the flags, subjects, and hours lists, while the loop accumulates values by subject.

## Report Function and Program Entry Point

`format_report()` converts structured results into display text. `main()` coordinates the full workflow and returns its data for later reuse or testing. The main guard marks direct program execution.

In [6]:
def format_report(summary, errors):
    report_lines = [
        "Weekly Study Report",
        f"Valid sessions: {summary['session_count']}",
        f"Total hours: {summary['total_hours']:.1f}",
        f"Mean session hours: {summary['mean_hours']:.1f}",
        f"Completion rate: {summary['completion_rate']:.1f}%",
        f"Completion label: {summary['completion_label']}",
        f"Subjects: {', '.join(summary['subjects'])}",
        f"Validation errors: {len(errors)}",
    ]
    return "\n".join(report_lines)

def main(raw_lines):
    sessions, errors = load_sessions(raw_lines)
    summary = summarize_week(sessions)
    report = format_report(summary=summary, errors=errors)
    print(report)
    if errors:
        print("Error details:", errors)
    return sessions, summary, errors

if __name__ == "__main__":
    program_sessions, program_summary, program_errors = main(raw_session_lines)

Weekly Study Report
Valid sessions: 5
Total hours: 9.0
Mean session hours: 1.8
Completion rate: 80.0%
Completion label: steady
Subjects: Data, Python, Statistics
Validation errors: 1
Error details: [(5, 'hours must be numeric')]


## Result Analysis

Five of six raw rows are valid. The invalid Friday row is recorded as `(5, 'hours must be numeric')`, and processing continues through Saturday. The valid sessions total 9.0 hours with a mean of 1.8 hours. Four of five sessions are complete, producing an 80.0% completion rate and the label `steady`. The subjects are Data, Python, and Statistics; `all_completed` is false and `any_incomplete` is true because Wednesday remains incomplete.

The program separates responsibilities cleanly: the class represents one session, the parser validates one row, the loader recovers across rows, the summarizer calculates results, the formatter creates text, and `main()` controls the complete workflow.

## Program Extension Points

The same structure can accept live `input()` values, lines read from a file, or rows received from another system. New validation rules belong in `parse_session()`, new summary fields belong in `summarize_week()`, and alternative output formats belong in separate formatter functions.